# Toxic Language Guardrail

**What is toxic language?** Insults, threats, obscenity, identity attacks and similar abusive content.

**How this guardrail helps:** it checks a user's message or an LLM's answer so abusive text can be removed or blocked before anyone sees it. The [Toxic Language](https://guardrailsai.com/hub/validator/guardrails/toxic_language) validator runs a local Detoxify model that scores text on seven labels: `toxicity`, `severe_toxicity`, `obscene`, `threat`, `insult`, `identity_attack` and `sexual_explicit`. If any score is above `threshold` (default `0.5`), validation fails.

**One-time setup** (already done in `myenv`, the kernel for this notebook):
```bash
pip install guardrails-ai guardrails-ai-toxic-language
python -m guardrails_ai.toxic_language.post_install   # downloads the NLTK data and Detoxify model
```

In [ ]:
import warnings

# silence harmless warnings from tqdm and guardrails' sync fallback in Jupyter
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="Could not obtain an event loop")

from dotenv import load_dotenv
from guardrails import Guard  ## the main wrapper for Guardrails
from guardrails.errors import ValidationError
from guardrails_ai.toxic_language import ToxicLanguage

# load the env vars
load_dotenv()

## 1. Create the validator and the guard

`Guard().use(...)` needs a validator **instance**, not the `Guard().use(ToxicLanguage, threshold=0.5, ...)` form shown in the docs.

`on_fail="fix"` removes the toxic sentences and keeps the rest.

In [ ]:
# create the validator
validator = ToxicLanguage(threshold=0.5, validation_method="sentence", on_fail="fix")

# create guard and add the validator to the guard
guard = Guard().use(validator)
guard.configure(allow_metrics_collection=False)

## 2. Validate user input

In [ ]:
user_input_1 = "Great job! You are doing amazing work. There were times when you got stuck but you still managed to get the work done. Kudos to you!"
user_input_2 = "Great job! You are doing amazing work. There were times when you got stuck but you still managed to get the work done. If you were not this stupid, the work would have been done a lot quicker!"

response_1 = guard.validate(user_input_1)

print(f"Validation Result: {response_1.validation_passed}")
print(f"Raw User Input: {response_1.raw_llm_output}")
print(f"Validated Output: {response_1.validated_output}")

With `on_fail="fix"`, `validation_passed` stays `True` even when toxic text was removed. To find out whether anything was flagged, check `validation_summaries`.

In [ ]:
response_2 = guard.validate(user_input_2)

print(f"Validation Result: {response_2.validation_passed}")
print(f"Toxic Language Found: {len(response_2.validation_summaries) > 0}")
print(f"Raw User Input: {response_2.raw_llm_output}")
print(f"Validated Output: {response_2.validated_output}")
print(f"Reason: {response_2.validation_summaries[0].error_spans[0].reason}")

## 3. Other ways to handle a failure

- `on_fail="exception"` raises a `ValidationError`.
- `validation_method="full"` scores the whole text, and with `fix` it returns an empty string if the text is toxic.
- `on_fail="reask"` fills in `response.reask` with details you can use to ask an LLM to try again. `guard.validate()` doesn't call an LLM itself, so it returns the fixed text.

In [ ]:
exception_guard = Guard().use(ToxicLanguage(on_fail="exception"))
exception_guard.configure(allow_metrics_collection=False)

try:
    exception_guard.validate(user_input_2)
except ValidationError as e:
    print(f"Blocked: {e}")

In [ ]:
full_guard = Guard().use(ToxicLanguage(validation_method="full", on_fail="fix"))
full_guard.configure(allow_metrics_collection=False)

print(f"Full Text Output: {full_guard.validate(user_input_2).validated_output!r}")

In [ ]:
reask_guard = Guard().use(ToxicLanguage(on_fail="reask"))
reask_guard.configure(allow_metrics_collection=False)

response_3 = reask_guard.validate(user_input_2)

print(f"Validated Output: {response_3.validated_output}")
print(f"Reask: {response_3.reask}")

## 4. What it can get wrong

The model reacts strongly to individual words, so test it on text from your own application:

- **Harmless text can be flagged:** technical uses of words like *kill* and *stupid* score high.
- **Polite hostility can pass:** an unwelcoming message with no rude words isn't flagged.
- **Disguised spelling can pass:** `stup1d` scores below `0.5`.

In [ ]:
examples = [
    "This bug is a stupid problem to have.",
    "I will kill this process and restart the server.",
    "Nobody wants you here, just leave.",
    "You are so stup1d.",
]

for text in examples:
    print(f"{text} -> {validator.get_toxicity(text) or 'not flagged'}")